# Video Face Swap — LTX 2.3 — ComfyUI

LTX 2.3 22B Dev FP8 ile videoda yüz değiştirme.

| | |
|---|---|
| **Ana Model** | LTX 2.3 22B Dev FP8 |
| **Custom Nodes** | ComfyUI-LTXVideo, ComfyUI-KJNodes, ComfyUI-VideoHelperSuite |

## Ön Hazırlık
- Colab Secrets: `HF_TOKEN`
- Opsiyonel: `CF_TUNNEL_TOKEN` (Cloudflare tunnel için)

## Kullanım
A: Kurulum → B: Model indir → C: Başlat → Browser'dan workflow yükle

---
# A) Kurulum

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

!pip install -q -U --pre comfyui-manager

# Custom Node'lar
NODES = {
    'ComfyUI-LTXVideo': 'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'rgthree-comfy': 'https://github.com/rgthree/rgthree-comfy.git',
    'ComfyUI-Custom-Scripts': 'https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git',
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-QwenVL': 'https://github.com/kijai/ComfyUI-QwenVL.git',
    'ComfyUI-MelBandRoformer': 'https://github.com/kijai/ComfyUI-MelBandRoformer.git',
    'bfsnodes': 'https://github.com/bluefoxcreation/bfsnodes.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# kornia fix for LTXVideo
!pip install -q kornia==0.7.3

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

In [ ]:
import shutil

from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    """HuggingFace'ten dosya indir. Mevcutsa atla."""
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Başarısız: {e}')

# Diffusion Model
print('\U0001f4e5 Diffusion Model:')
hf_download('Kijai/LTX2.3_comfy', 'diffusion_models/ltx-2.3-22b-dev_transformer_only_fp8_scaled.safetensors', f'{MODELS_DIR}/diffusion_models')

# Text Encoders
print('\n\U0001f4e5 Text Encoders:')
hf_download('Comfy-Org/ltx-2', 'split_files/text_encoders/gemma_3_12B_it.safetensors', f'{MODELS_DIR}/text_encoders')
hf_download('Kijai/LTX2.3_comfy', 'text_encoders/ltx-2-3-22b-text_encoder.safetensors', f'{MODELS_DIR}/text_encoders')

# VAE
print('\n\U0001f4e5 VAE:')
hf_download('Kijai/LTX2.3_comfy', 'vae/LTX23_video_vae_bf16.safetensors', f'{MODELS_DIR}/vae')
import os
vae_src = f'{MODELS_DIR}/vae/LTX23_video_vae_bf16.safetensors'
vae_link = f'{MODELS_DIR}/vae/ltx-2-3-22b-VAE.safetensors'
if os.path.exists(vae_src) and not os.path.exists(vae_link):
    os.symlink(vae_src, vae_link)
hf_download('Kijai/LTX2.3_comfy', 'vae/LTX23_audio_vae_bf16.safetensors', f'{MODELS_DIR}/vae')
audio_src = f'{MODELS_DIR}/vae/LTX23_audio_vae_bf16.safetensors'
audio_link = f'{MODELS_DIR}/vae/ltx-2-3-22b-audio_vae.safetensors'
if os.path.exists(audio_src) and not os.path.exists(audio_link):
    os.symlink(audio_src, audio_link)

# Latent Upscale
print('\n\U0001f4e5 Latent Upscale:')
hf_download('Lightricks/LTX-2.3', 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors', f'{MODELS_DIR}/latent_upscale_models')

# LoRA
print('\n\U0001f4e5 LoRA:')
hf_download('Kijai/LTX2.3_comfy', 'loras/head_swap_v3_rank_adaptive_fro_098.safetensors', f'{MODELS_DIR}/loras')
hf_download('Lightricks/LTX-2.3', 'ltx-2.3-22b-distilled-lora-384.safetensors', f'{MODELS_DIR}/loras')

# MelBandRoformer (audio separation)
print('\n\U0001f4e5 Audio Separation:')
hf_download('Kim2091/open-unmix-pytorch', 'MelBandRoformer_fp32.safetensors', f'{MODELS_DIR}/audio_separation')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*', '--enable-manager'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI çöktü!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)